# Section 1: Dataset Understanding & Structural Integrity

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pathlib import Path

load_dotenv()

current_dir = Path.cwd().name

if  current_dir == "notebook":
    os.chdir("..")
# 2. Configure high-fidelity pandas viewing options
pd.set_option('display.max_columns', None)

DB_PATH = os.getenv("DB_PATH", "../data/ml_project.db")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM raw_data", conn)
conn.close()

print(f"Total Patient Records: {df.shape[0]}")
print(f"Total Initial Features: {df.shape[1]}")


In [ ]:
print("=== Component Metadata Layout ===")
print(df.dtypes)

print("\n=== Initial Matrix Sample View ===")
display(df.head(5))

## Heart Disease Dataset Features (Compact)

- age: Age of the patient (years)
- sex: Gender (1 = male, 0 = female)
- cp: Type of chest pain
- trestbps: Resting blood pressure (mm Hg)
- chol: Serum cholesterol (mg/dl)
- fbs: Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- restecg: Resting ECG results
- thalach: Maximum heart rate achieved
- exang: Exercise-induced angina (1 = yes, 0 = no)
- oldpeak: ST depression induced by exercise
- slope: Slope of peak exercise ST segment
- ca: Number of major vessels colored by fluoroscopy
- thal: Thalassemia type
- num: Heart disease presence (target variable)

In [ ]:
# mark all ? as nan
df.replace('?', np.nan, inplace=True)

# numerical features
numerical_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

for col in numerical_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# target feature  to a numeric type
df['num'] = pd.to_numeric(df['num'], errors='coerce')

print("===  Missing Value  ===")
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

missing_values = pd.DataFrame({
    "Raw Missing Count": null_counts,
    "Percentage (%)": null_percentages,
    "Current DataType": df.dtypes
}).sort_values(by="Raw Missing Count", ascending=False)

# Display only features containing actual missing values
display(missing_values[missing_values["Raw Missing Count"] > 0])

In [ ]:
duplicate_records = df.duplicated().sum()
print(f"Data duplication check: Found {duplicate_records} identical rows ")

if duplicate_records > 0:
    df.drop_duplicates(inplace=True)
    print(f"removed duplicates: {df.shape[0]} unique rows remain.")
else:
    print("no duplicates found...")

In [ ]:
print("=== raw multi-Class severity scales ===\n")
print(df['num'].value_counts().sort_index())

# create a binary target variable: 1 for presence of heart disease (num > 0), 0 for absence (num = 0)
df['target'] = df['num'].apply(lambda x: 1 if x > 0 else 0)

print("\n=== Operational Binary Stratification Balance ===\n")
raw_counts = df['target'].value_counts()
proportions = df['target'].value_counts(normalize=True) * 100

for label, count in raw_counts.items():
    state = "Healthy (0)" if label == 0 else "Sick (1)"
    print(f"{state}: {count} patients ({proportions[label]:.2f}%)")


In [ ]:
# Numerical Feature Summary
display(df[numerical_features].describe().T)

# Section 2: Detail-Driven Feature Analysis

In [ ]:
# Configure premium, publication-grade visualization formatting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'figure.titlesize': 18,
    'figure.figsize': (12, 8)
})

In [ ]:
# kde plots for numerical features stratified by target class
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_features):
    # Smooth distribution lines (Green = Healthy, Red = Sick)
    sns.kdeplot(data=df[df['target']==0], x=col, fill=True, color="#488f31", alpha=0.4, label="Healthy", ax=axes[idx], linewidth=2)
    sns.kdeplot(data=df[df['target']==1], x=col, fill=True, color="#de425b", alpha=0.4, label="Sick", ax=axes[idx], linewidth=2)
    
    # Calculate and plot demographic medians
    healthy_med = df[df['target']==0][col].median()
    sick_med = df[df['target']==1][col].median()
    axes[idx].axvline(healthy_med, color="darkgreen", linestyle="--", alpha=0.8, label=f"Healthy Med: {healthy_med:.1f}")
    axes[idx].axvline(sick_med, color="darkred", linestyle="--", alpha=0.8, label=f"Sick Med: {sick_med:.1f}")
    
    axes[idx].set_title(f"Density Shift: {col.upper()}", weight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].legend(fontsize=10)

fig.delaxes(axes[5]) # Remove the blank 6th grid box to keep the layout crisp
plt.suptitle("Stratified Continuous Feature Topography (KDE Analysis)", weight='bold', fontsize=20, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# outlier dynamics and variance landscape
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Map a temporary human-readable label for clean chart text
df['Status_Label'] = df['target'].map({0: 'Healthy', 1: 'Sick'})

for idx, col in enumerate(numerical_features):
    # Violin plots trace distribution thickness boundaries
    sns.violinplot(data=df, x="Status_Label", y=col,hue="Status_Label", palette={"Healthy": "#488f31", "Sick": "#de425b"}, 
                   ax=axes[idx], inner="box", alpha=0.5, cut=0)
    # Strip plots scatter the raw patient coordinates on top
    sns.stripplot(data=df, x="Status_Label", y=col, color="black", alpha=0.2, jitter=0.25, ax=axes[idx], size=4)
    
    axes[idx].set_title(f"Variance Landscape: {col.capitalize()}", weight='bold')
    axes[idx].set_xlabel("Clinical Diagnosis State")
    axes[idx].set_ylabel(col)

fig.delaxes(axes[5])
plt.suptitle("Clinical Variance Landscapes & Outlier Boundary Verification", weight='bold', fontsize=20, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Discrete Risk Factors & Categorical Layouts
categorical_features = ["cp", "restecg", "slope", "thal"]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, col in enumerate(categorical_features):
    sns.countplot(data=df, x=col, hue="Status_Label", palette={"Healthy": "#488f31", "Sick": "#de425b"}, ax=axes[idx])
    
    axes[idx].set_title(f"Categorical Risk Profile: [{col.upper()}]", weight='bold')
    axes[idx].set_xlabel(f"Dataset Code for {col}")
    axes[idx].set_ylabel("Patient Count")
    axes[idx].legend(title="Diagnosis", loc="upper right")

plt.suptitle("Categorical Frequency Distributions", weight='bold', fontsize=20, y=0.96)
plt.tight_layout()
plt.show()

In [ ]:
# Master Correlation Topology
'''Standard Pearson correlation matrix tracks only basic numbers, missing how 
separate categorical states affect things. To map everything perfectly, we 
temporarily unpack all categorical attributes into explicit one-hot encoded
binary matrices. We mask the upper symmetric half to make it incredibly
easy to scan for multi-collinearity.
'''
# Create a copy of our continuous parameters
correlation_matrix_df = df[numerical_features].copy()
correlation_matrix_df['Operational_Target'] = df['target']

# Explode categorical attributes into binary columns
for col in categorical_features:
    # Use cast to int to ensure numeric 1s and 0s are added to the matrix
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=False, dtype=int)
    correlation_matrix_df = pd.concat([correlation_matrix_df, dummies], axis=1)
    
# Generate a structural lower-triangle mask
plt.figure(figsize=(22, 18))
structural_mask = np.triu(np.ones_like(correlation_matrix_df.corr(), dtype=bool))

# Render the high-resolution visualization matrix
sns.heatmap(
    correlation_matrix_df.corr(), 
    annot=True, 
    fmt=".2f", 
    cmap="vlag", 
    mask=structural_mask, 
    linewidths=0.7, 
    annot_kws={"size": 10}, 
    cbar_kws={"shrink": .8}
)

plt.title("Master Clinical Correlation Matrix Topology", fontsize=20, weight='bold', pad=25)
plt.show()

# Section 3: Preprocessing Blueprinting & Data Warehouse Isolation

In [ ]:
# Drop columns that are temporary visualization/unaggregated targets
X = df.drop(columns=['num', 'target', 'Status_Label'], errors='ignore')
y = df['target']

# Isolate columns dynamically by their structural preprocessing tracks
numerical_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features = [col for col in X.columns if col not in numerical_features]

# Perform Stratified Train/Test Split (Preserves the exact healthy/sick ratio in both splits)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("=== Validation Boundary Split Metrics ===\n")
print(f"Training Feature Set Space: {X_train.shape}")
print(f"Testing Evaluation Space: {X_test.shape}")
print(f"Stratified Base Split Target Ratio: {y_train.value_counts(normalize=True).values[1]*100:.1f}% Sick")

In [ ]:
# 1. Build the separate pipelines for different data types
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())  # Outlier-resistant mathematical scaling
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 2. Pack the pipelines into a single, comprehensive ColumnTransformer object
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# 3. Fit on training matrices and run a validation transformation test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Reconstruct a preview dataframe to verify structural integrity
processed_columns = (numerical_features + 
                     list(preprocessor.named_transformers_['cat']
                          .named_steps['encoder'].get_feature_names_out(categorical_features)))

df_processed_preview = pd.DataFrame(X_train_processed, columns=processed_columns)

print(f"Processed Feature Matrix Shape: {X_train_processed.shape[1]} columns.")
display(df_processed_preview.head(5))

In [ ]:
# save dataframes to csv for later use in modeling

CLEANSED_DB_PATH = os.getenv("CLEANSED_DB_PATH", "data/cleansed.db")

if os.path.dirname(CLEANSED_DB_PATH):
    os.makedirs(os.path.dirname(CLEANSED_DB_PATH), exist_ok=True)
    
# Save using a connection manager with an explicit timeout to prevent file lock drops
with sqlite3.connect(CLEANSED_DB_PATH, timeout=15) as cleansed_conn:
    print(f"Committing curated matrix to warehouse at: {CLEANSED_DB_PATH}...")
    df.to_sql("heart_disease_clean", cleansed_conn, if_exists="replace", index=False)
    cleansed_conn.commit()
    
# Run an isolated verify read to confirm table rows are perfectly accessible
with sqlite3.connect(CLEANSED_DB_PATH, timeout=15) as verification_conn:
    df_check = pd.read_sql_query("SELECT * FROM heart_disease_clean", verification_conn)
    
print(f"Data Lineage Complete! Verified Records: {df_check.shape[0]} rows safely locked in.")